In [8]:
# Exploración de la API de eBird
#
# La API usa códigos jerárquicos de región:
# - world devuelve países.
# - Un país, por ejemplo CL, devuelve sus regiones administrativas.
# - Una región, por ejemplo CL-RM, puede devolver sus subregiones.
#
# Las observaciones recientes se consultan con /data/obs/{regionCode}/recent.
# La API no entrega directamente archivos de foto o audio; hasRichMedia
# indica que puede existir multimedia asociada a la observación.

## Qué entrega esta consulta

Cada observación puede incluir especie (`comName`, `sciName`), fecha (`obsDt`), lugar (`locName`), cantidad (`howMany`), coordenadas (`lat`, `lng`) y, según el endpoint y la observación, `hasRichMedia`.

Para ordenar las aves más comunes habría que agrupar por `comName` y contar observaciones o sumar `howMany`. Para fotos y cantos habrá que consultar una fuente multimedia aparte; esta API solo permite detectar que puede existir contenido enriquecido, no descargarlo directamente.

In [7]:
# Resumen útil para una futura visualización.
import pandas as pd

observations_df = pd.DataFrame(observations)
summary_columns = [
    "comName",
    "sciName",
    "obsDt",
    "locName",
    "howMany",
    "lat",
    "lng",
    "hasRichMedia",
]
summary = observations_df.reindex(columns=summary_columns)
summary.head(20)

,comName,sciName,obsDt,locName,howMany,lat,lng,hasRichMedia
0,Diuca Finch,Diuca diuca,2026-09-17 12:43,Parquemet--Cerro San Cristóbal--Sector Tupahue,2,-33.415886,-70.622736,NaN
1,Long-tailed Meadowlark,Leistes loyca,2026-09-17 12:43,Parquemet--Cerro San Cristóbal--Sector Tupahue,3,-33.415886,-70.622736,NaN
2,Austral Blackbird,Curaeus curaeus,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",2,-33.400910,-71.229271,NaN
3,White-crested Elaenia,Elaenia albiceps,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",2,-33.400910,-71.229271,NaN
4,Chilean Mockingbird,Mimus thenca,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",6,-33.400910,-71.229271,NaN
5,California Quail,Callipepla californica,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",7,-33.400910,-71.229271,NaN
6,Rufous-collared Sparrow,Zonotrichia capensis,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",8,-33.400910,-71.229271,NaN
7,Chimango Caracara,Daptrius chimango,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",1,-33.400910,-71.229271,NaN
8,Southern House Wren,Troglodytes musculus,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",1,-33.400910,-71.229271,NaN
9,Shiny Cowbird,Molothrus bonariensis,2026-09-17 12:18,Parque Bicentenario de Cerrillos,19,-33.493379,-70.699511,NaN


In [9]:
common_species = (
    observations_df.groupby(["speciesCode", "comName", "sciName"], dropna=False)
    .agg(
        observaciones=("speciesCode", "size"),
        individuos_reportados=("howMany", "sum"),
        lugares=("locId", "nunique"),
        ultima_observacion=("obsDt", "max"),
    )
    .sort_values(["observaciones", "individuos_reportados"], ascending=False)
    .reset_index()
)

common_species.head(20)

,speciesCode,comName,sciName,observaciones,individuos_reportados,lugares,ultima_observacion
0,eardov1,Eared Dove,Zenaida auriculata,1,46,1,2026-09-17 12:18
1,gryfin2,Greater Yellow-Finch,Sicalis auriventris,1,45,1,2026-09-16 16:22
2,andgoo1,Andean Goose,Oressochen melanopterus,1,41,1,2026-09-16 09:42
3,brhgul2,Brown-hooded Gull,Chroicocephalus maculipennis,1,32,1,2026-09-16 09:42
4,yebpin1,Yellow-billed Pintail,Anas georgica,1,26,1,2026-09-16 10:02
5,bknsti,Black-necked Stilt,Himantopus mexicanus,1,24,1,2026-09-16 09:42
6,gryfin1,Grassland Yellow-Finch,Sicalis luteola,1,24,1,2026-09-17 12:18
7,categr1,Western Cattle-Egret,Ardea ibis,1,20,1,2026-09-16 09:42
8,shicow,Shiny Cowbird,Molothrus bonariensis,1,19,1,2026-09-17 12:18
9,regcoo1,Red-gartered Coot,Fulica armillata,1,12,1,2026-09-16 11:27


In [6]:
# Consulta de observaciones recientes de una región.
# Cambia este código por cualquiera de chile_regions.
REGION_CODE = "CL-RM"

observations = ebird_get(
    f"/data/obs/{REGION_CODE}/recent",
    params={
        "back": 7,
        "maxResults": 100,
        "detail": "full",
        "includeProvisional": "true",
    },
)

print(f"Observaciones recibidas para {REGION_CODE}: {len(observations)}")
observations[:3]

Observaciones recibidas para CL-RM: 100


[{'speciesCode': 'codfin1',
  'comName': 'Diuca Finch',
  'sciName': 'Diuca diuca',
  'locId': 'L9990332',
  'locName': 'Parquemet--Cerro San Cristóbal--Sector Tupahue',
  'obsDt': '2026-09-17 12:43',
  'howMany': 2,
  'lat': -33.4158865,
  'lng': -70.6227357,
  'obsValid': True,
  'obsReviewed': False,
  'locationPrivate': False,
  'subId': 'S393688912'},
 {'speciesCode': 'lotmea1',
  'comName': 'Long-tailed Meadowlark',
  'sciName': 'Leistes loyca',
  'locId': 'L9990332',
  'locName': 'Parquemet--Cerro San Cristóbal--Sector Tupahue',
  'obsDt': '2026-09-17 12:43',
  'howMany': 3,
  'lat': -33.4158865,
  'lng': -70.6227357,
  'obsValid': True,
  'obsReviewed': False,
  'locationPrivate': False,
  'subId': 'S393688912'},
 {'speciesCode': 'ausbla1',
  'comName': 'Austral Blackbird',
  'sciName': 'Curaeus curaeus',
  'locId': 'L77312056',
  'locName': 'Curacavi, Santiago Metropolitan Region, CL (-33.401, -71.229)',
  'obsDt': '2026-09-17 12:28',
  'howMany': 2,
  'lat': -33.4009098,
  'l

In [5]:
# Regiones administrativas de Chile.
chile_regions = show_region_options("subnational1", "CL")
chile_regions

[{'code': 'CL-AI', 'name': 'Aisén del General Carlos Ibáñez del Campo'},
 {'code': 'CL-AN', 'name': 'Antofagasta'},
 {'code': 'CL-AR', 'name': 'Araucanía'},
 {'code': 'CL-AP', 'name': 'Arica y Parinacota'},
 {'code': 'CL-AT', 'name': 'Atacama'},
 {'code': 'CL-BI', 'name': 'Bío-Bío'},
 {'code': 'CL-CO', 'name': 'Coquimbo'},
 {'code': 'CL-LI', 'name': "Libertador General Bernardo O'Higgins"},
 {'code': 'CL-LL', 'name': 'Los Lagos'},
 {'code': 'CL-LR', 'name': 'Los Ríos'},
 {'code': 'CL-MA', 'name': 'Magallanes'},
 {'code': 'CL-ML', 'name': 'Maule'},
 {'code': 'CL-RM', 'name': 'Región Metropolitana de Santiago'},
 {'code': 'CL-TA', 'name': 'Tarapacá'},
 {'code': 'CL-VS', 'name': 'Valparaíso'},
 {'code': 'CL-NB', 'name': 'Ñuble'}]

In [3]:
def ebird_get(path, params=None):
    response = requests.get(
        f"{BASE_URL}{path}",
        headers={"X-eBirdApiToken": EBIRD_API_KEY},
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def show_region_options(region_type, parent_code):
    """List regions below a world, country, or subnational1 parent."""
    return ebird_get(f"/ref/region/list/{region_type}/{parent_code}")


# Países disponibles para consultar.
world_regions = show_region_options("country", "world")
world_regions[:10]

[{'code': 'AF', 'name': 'Afghanistan'},
 {'code': 'AL', 'name': 'Albania'},
 {'code': 'DZ', 'name': 'Algeria'},
 {'code': 'AS', 'name': 'American Samoa'},
 {'code': 'AD', 'name': 'Andorra'},
 {'code': 'AO', 'name': 'Angola'},
 {'code': 'AI', 'name': 'Anguilla'},
 {'code': 'AQ', 'name': 'Antarctica'},
 {'code': 'AG', 'name': 'Antigua and Barbuda'},
 {'code': 'AR', 'name': 'Argentina'}]

In [1]:
from pathlib import Path
import os

import requests

BASE_URL = "https://api.ebird.org/v2"


def load_env_value(name):
    """Load one simple KEY=value entry without printing the secret."""
    value = os.getenv(name)
    if value:
        return value

    env_path = Path.cwd() / ".env"
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            key, separator, candidate = line.partition("=")
            if separator and key.strip() == name:
                return candidate.strip().strip('"').strip("'")
    return None


EBIRD_API_KEY = (
    load_env_value("EBIRD_API_KEY")
    or load_env_value("API_BIRD_KEY")
    or load_env_value("X_EBIRDAPITOKEN")
)

if not EBIRD_API_KEY:
    raise RuntimeError("No se encontró EBIRD_API_KEY, API_BIRD_KEY o X_EBIRDAPITOKEN en .env")

print("Clave cargada: sí")
print("Directorio de trabajo:", Path.cwd())

Clave cargada: sí
Directorio de trabajo: /Users/jabac/Documents/universidad/vi-semestre/,infovis/proyecto
